In [ ]:
# Ensure gradalg is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# gradalg isn't already installed into this kernel.
try:
    import gradalg  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "gradalg" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import gradalg  # noqa: F401

# 07 — Derived Bracket

Bu notebook [07_derived_bracket.md](07_derived_bracket.md) markdown'ının çalıştırılabilir sürümüdür. `{a, b}_Q := [[a, Q]_base, b]_base` inşası, Jacobi'nin tek bir denkleme (`[Q, Q]_base = 0`) indirilmesi, Koszul eşdeğerliği, Poisson ve H-twisted Courant köşeleri.

## İnşa

`DerivedBracket(base, Q, degree_Q=...)` — Lie base üstünde degree-1 `Q` seçelim. `|{·,·}_Q| = |Q| − 2 = −1`. Leibniz evrensel, antisymmetry/Jacobi koşullu (flag=None).

In [ ]:
from gradalg.brackets.derived import DerivedBracket
from gradalg.brackets.lie import LieBracket
from gradalg.core.expr import Symbol
from gradalg.core.properties import Graded
from gradalg.core.registry import PropertyRegistry

reg = PropertyRegistry()
Q = Symbol('Q')
reg.declare(Q, Graded(degree=1))
lie = LieBracket()

d = DerivedBracket(lie, Q, degree_Q=1)
print('name   :', d.name)
print('degree :', d.degree)
print('antisym:', d.is_graded_antisymmetric,
      'leibniz:', d.satisfies_leibniz,
      'jacobi :', d.satisfies_graded_jacobi)

## İki yüzlü expansion

`expand` — iç/dış iki katman base açılmış; `expand_definition` — iki katman base `BracketApply` inert.

In [ ]:
a, b = Symbol('a'), Symbol('b')
for s in (a, b):
    reg.declare(s, Graded(degree=0))

print('expand           :', d.expand(a, b, reg))
print('expand_definition:', d.expand_definition(a, b, reg))

## Jacobi obstruction — üç yüz

Tek bir koşul: `[Q, Q]_base = 0`. Lie base için trivial (`Q*Q − Q*Q`).

In [ ]:
print('expanded :', d.jacobi_obstruction(reg))
print('raw      :', d.jacobi_obstruction_raw())
cond = d.jacobi_condition(reg)
print('condition:', cond.name)
print('holds?   :', cond.holds(reg))

## `prove_jacobi` — DerivedBracketStrategy

Bracket tipinden otomatik dispatch. Üç adım: DerivedBracketTheorem → base-bracket-expand → simplify.

In [ ]:
from gradalg.proof.verifier import prove_jacobi

# a, b zaten yukarıda Graded(0) olarak kayıtlı — sadece c'yi ekle.
c = Symbol('c')
reg.declare(c, Graded(degree=0))

chain = prove_jacobi(d, a, b, c, registry=reg)
print('chain len:', len(chain))
for st in chain.steps:
    print(' ', st.rule)
print('final:', chain.steps[-1].after)

## `acting_on` — Koszul eşdeğerliği

SN base + π generator + anchor ρ: `expand` otomatik olarak Koszul 3-terim formunu emit eder. `KoszulBracket(ρ)` ile structurally eşit.

In [ ]:
from gradalg.brackets.schouten import sn
from gradalg.brackets.koszul import KoszulBracket
from gradalg.calculus.anchor import Anchor

reg2 = PropertyRegistry()
pi = Symbol('π')
reg2.declare(pi, Graded(degree=1))
alpha, beta = Symbol('α'), Symbol('β')
for s in (alpha, beta):
    reg2.declare(s, Graded(degree=1))

rho = Anchor('ρ')
koszul_derived = DerivedBracket(sn, pi, degree_Q=1, acting_on=rho)
koszul_classical = KoszulBracket(rho)

lhs = koszul_derived.expand(alpha, beta)
rhs = koszul_classical.expand(alpha, beta)
print('derived :', lhs)
print('classic :', rhs)
print('equal?  :', lhs == rhs)

## Poisson-as-derived — library wrapper

Matematiksel olarak `DerivedBracket(sn, π, degree_Q=1)` Poisson bracket. `[π, π]_SN`'nin SN-atomik olması sebebiyle `prove_jacobi` generic simplify yolu kapanmaz; `PoissonBracket.prove_jacobi_reduction` seeded teorem citation'ı ile bir adımda biter.

In [ ]:
from gradalg.library import theorem_book
from gradalg.library.declarations import Bivector, Functions
from gradalg.library.poisson import PoissonBracket

reg3 = PropertyRegistry()
pi3 = Bivector('π', registry=reg3)
f, g, h = Functions('f g h', degree=-1, registry=reg3)

poisson = PoissonBracket.from_bivector(pi3)
chain = poisson.prove_jacobi_reduction(f, g, h, registry=reg3)
print('reduction chain len:', len(chain))
print('  rule  :', chain.steps[0].rule)
print('  after :', chain.steps[0].after)

thm = theorem_book.get('poisson_jacobi')
print('theorem from_axioms:', thm.from_axioms)

## H-twist — Courant koşullu Jacobi

`CourantBracket(background_H=H)`: Jacobi ⟺ dH = 0. Default (H=None) vacuous.

In [ ]:
from gradalg.brackets.courant import CourantBracket

reg4 = PropertyRegistry()
H = Symbol('H')
reg4.declare(H, Graded(degree=3))

print('untwisted:', CourantBracket().jacobi_condition(reg4).name)

C = CourantBracket(background_H=H)
print('twisted  :', C.is_twisted)
cond = C.jacobi_condition(reg4)
print('  name       :', cond.name)
print('  obstruction:', cond.obstruction)

## Sonraki adım

Stage D: birleşik tur + foundations — [08_unified_picture.md](08_unified_picture.md).